# Stress Prediction v24

**Root-cause fix for v23's LB 0.26 collapse.**

| Problem in v23 | Fix in v24 |
|---|---|
| Nelder-Mead alpha over-fit LOPO set | **Grid search** with hard distribution constraint |
| Session smoothing during calibration pushed class-1 to 264 | **No smoothing** during calibration search |
| BA maximization gameable by class-1 inflation | **Distribution constraint**: each class within 1.5× of prior |
| Single calibration method | **Threshold tuning** (primary) + alpha grid (secondary) + raw argmax (safety net) |

## 0. Install Dependencies

In [1]:
# Install required libraries
%pip install numpy pandas scipy scikit-learn lightgbm

# Optional: XGBoost for ensemble blending (60% LGBM / 40% XGB)
%pip install xgboost


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#!/usr/bin/env python3
"""
Stress Prediction v24 — Root-cause fix for v23's LB 0.26 collapse
======================================================================
DIAGNOSIS of v23 failure:
  - LOPO BA drops  0.5546 → 0.5333 AFTER calibration → calibration is HURTING
  - alpha_opt = [1.64, 0.44, 2.63] pushed class-1 to 264 (target ~83)
  - Nelder-Mead over-fit the LOPO OOF set; test distribution ≠ LOPO distribution
  - Session smoothing (strength=0.30) is blending wrong-direction probabilities

FIXES in v24:
  1. NO SESSION SMOOTHING during calibration search (was the hidden culprit)
  2. Calibration target: directly penalize distance from prior counts
     instead of maximizing BA (BA can be gamed by class-1 over-inflation)
  3. Grid search alpha instead of Nelder-Mead → no local optima, reproducible
  4. Hard constraint: calibrated class dist must stay within 1.5× of prior
  5. Threshold tuning on LOPO OOF instead of prior^alpha (more direct)
  6. Save multiple submissions including raw argmax (no calibration) as safety net

SUBMISSION STRATEGY:
  - submission_raw.csv        ← raw LGBM+XGB argmax, no calibration (safest)
  - submission_thresh.csv     ← threshold-tuned on LOPO OOF (balanced)
  - submission_opt.csv        ← constrained alpha grid search
  - submission.csv            ← copy of best expected (thresh)
"""

import warnings
import shutil
from pathlib import Path
from collections import Counter
from itertools import product

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut

import lightgbm as lgb

try:
    import xgboost as xgb
    HAS_XGB = True
    print("✓ XGBoost available — will blend LGBM + XGB")
except ImportError:
    HAS_XGB = False
    print("✗ XGBoost not available — using LightGBM only")

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

✓ XGBoost available — will blend LGBM + XGB


## 1. LOAD & CLEAN

In [3]:
DATA_DIR    = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes:')
print(f'  TRAIN_DATA : {TRAIN_DATA.shape}')
print(f'  TRAIN_LABEL: {TRAIN_LABEL.shape}')
print(f'  TEST_DATA  : {TEST_DATA.shape}')
print(f'  TEST_LABEL : {TEST_LABEL.shape}')

SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']


def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x']     = out['accel_x'].clip(-128, 127)
    out['accel_y']     = out['accel_y'].clip(-128, 127)
    out['accel_z']     = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)


def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out


TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned ✓')

Raw shapes:
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)
Cleaned ✓


## 2. RESTING BASELINE PER SUBJECT

In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr   = grp['heart_rate'].values.astype(float)
        eda  = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {
                'hr': float(np.nanmedian(hr)), 'eda': float(np.nanmedian(eda)),
                'temp': float(np.nanmedian(temp)), 'hr_std': 5.0, 'eda_std': 0.5
            }
            continue
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z  = (hr_v  - hr_v.mean())  / (hr_v.std()  + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal = hr_z + eda_z
        thr  = np.percentile(arousal, low_pct)
        mask = arousal < thr
        if mask.sum() < 10:
            mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':      float(np.median(hr_v[mask])),
            'eda':     float(np.median(eda_v[mask])),
            'temp':    float(np.median(temp_v[mask])),
            'hr_std':  float(np.std(hr_v[mask])  + 1e-3),
            'eda_std': float(np.std(eda_v[mask]) + 1e-3),
        }
    return refs


TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)
print(f'Resting baselines: {len(TRAIN_REFS)} train / {len(TEST_REFS)} test PIDs ✓')

Resting baselines: 7 train / 8 test PIDs ✓


## 3. SESSION TEMPORAL ENRICHMENT

In [5]:
GAP_MS = 30 * 60 * 1000


def enrich_labels_with_session_info(label_df):
    out = label_df.copy()
    for col in ['session_elapsed_ms', 'session_position', 'session_label_idx', 'session_total_labels']:
        out[col] = np.nan
    for pid, grp in out.groupby('pid'):
        grp_s  = grp.sort_values('timestamp')
        ts     = grp_s['timestamp'].values
        breaks = np.where(np.diff(ts) > GAP_MS)[0] + 1
        bounds = np.r_[0, breaks, len(ts)]
        for i in range(len(bounds) - 1):
            s, e    = bounds[i], bounds[i + 1]
            idx     = grp_s.index[s:e]
            sess_ts = ts[s:e]
            elapsed = (sess_ts - sess_ts[0]).astype(float)
            dur     = float(elapsed[-1]) if len(elapsed) > 1 else 1.0
            out.loc[idx, 'session_elapsed_ms']   = elapsed
            out.loc[idx, 'session_position']     = elapsed / max(dur, 1.0)
            out.loc[idx, 'session_label_idx']    = np.arange(e - s, dtype=float)
            out.loc[idx, 'session_total_labels'] = float(e - s)
    return out


TRAIN_LABEL_E = enrich_labels_with_session_info(TRAIN_LABEL)
TEST_LABEL_E  = enrich_labels_with_session_info(TEST_LABEL)
print('Session temporal info added ✓')

Session temporal info added ✓


## 4. FEATURE EXTRACTION

In [6]:
WINDOW_MS = 180_000
HALF_MS   =  90_000
THIRD_MS  =  60_000
SHORT_MS  =  60_000
LAG_MS    = 180_000


def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn', 'rmssd', 'pnn25', 'pnn50', 'mean_rr', 'cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr      = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25) * 100) if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50) * 100) if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f


def hrv_frequency_domain(bpm_series):
    f = {'hrv_lf': np.nan, 'hrv_hf': np.nan, 'hrv_lf_hf': np.nan, 'hrv_total_power': np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30:
        return f
    rr   = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_c = rr - rr.mean()
    nperseg = min(len(rr_c), 64)
    if nperseg < 16:
        return f
    try:
        freqs, psd = sps.welch(rr_c, fs=1.0, nperseg=nperseg, noverlap=nperseg // 2)
        def bp(lo, hi):
            m = (freqs >= lo) & (freqs < hi)
            return float(trapezoid(psd[m], freqs[m])) if m.sum() >= 2 else 0.0
        f['hrv_lf']          = bp(0.04, 0.15)
        f['hrv_hf']          = bp(0.15, 0.40)
        f['hrv_total_power'] = bp(0.0033, 0.40)
        f['hrv_lf_hf']       = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except Exception:
        pass
    return f


def eda_peak_features(eda_series):
    f = {'eda_n_peaks': np.nan, 'eda_peaks_per_min': np.nan,
         'eda_mean_prominence': np.nan, 'eda_max_prominence': np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40:
        return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 16:
        return f
    try:
        wl = min(15, len(eda_4hz) // 2 * 2 + 1)
        phasic = (eda_4hz - sps.savgol_filter(eda_4hz, wl, 2) + np.mean(eda_4hz)
                  if wl >= 5 else eda_4hz)
        peaks, props = sps.find_peaks(phasic, prominence=0.02, distance=4, width=1)
        f['eda_n_peaks']         = float(len(peaks))
        dur_min                  = len(eda_4hz) / (4.0 * 60.0)
        f['eda_peaks_per_min']   = float(len(peaks) / dur_min) if dur_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
        else:
            f['eda_mean_prominence'] = f['eda_max_prominence'] = 0.0
    except Exception:
        pass
    return f


def extract_features(label_df_e, sensor_df, pid_enc_map, refs):
    sensor_by_pid = {
        pid: grp.sort_values('timestamp').reset_index(drop=True)
        for pid, grp in sensor_df.groupby('pid')
    }
    rows = []
    for n, lrow in enumerate(label_df_e.itertuples(index=False), 1):
        pid  = lrow.pid
        ts   = float(lrow.timestamp)
        feat = {'id': int(lrow.id)}

        for col in ['session_elapsed_ms', 'session_position', 'session_label_idx', 'session_total_labels']:
            val = getattr(lrow, col)
            feat[col] = float(val) if np.isfinite(float(val)) else np.nan

        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat)
            continue

        ta    = sg['timestamp'].values
        wa    = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts),              SENSOR_COLS]
        wf    = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS),    SENSOR_COLS]
        wl    = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),             SENSOR_COLS]
        wt1   = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - 2*THIRD_MS), SENSOR_COLS]
        wt3   = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),             SENSOR_COLS]
        wshrt = sg.loc[(ta >= ts - SHORT_MS)  & (ta <= ts),             SENSOR_COLS]
        wlag  = sg.loc[(ta >= ts - WINDOW_MS - LAG_MS) & (ta < ts - WINDOW_MS), SENSOR_COLS]

        feat['window_count'] = len(wa)

        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            null_keys = ['mean','std','min','max','median','skew','kurt','range',
                         'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1','p10','p90']
            if len(v) == 0:
                for s in null_keys:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']    = float(np.mean(v))
            feat[f'{c}_std']     = float(np.std(v))
            feat[f'{c}_min']     = float(np.min(v))
            feat[f'{c}_max']     = float(np.max(v))
            feat[f'{c}_median']  = float(np.median(v))
            feat[f'{c}_skew']    = float(spstats.skew(v))     if len(v) > 2 else 0.0
            feat[f'{c}_kurt']    = float(spstats.kurtosis(v)) if len(v) > 2 else 0.0
            feat[f'{c}_range']   = float(np.max(v) - np.min(v))
            feat[f'{c}_q25']     = float(np.percentile(v, 25))
            feat[f'{c}_q75']     = float(np.percentile(v, 75))
            feat[f'{c}_p10']     = float(np.percentile(v, 10))
            feat[f'{c}_p90']     = float(np.percentile(v, 90))
            feat[f'{c}_iqr']     = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta']   = (float(np.mean(vl) - np.mean(vf))
                                    if len(vf) and len(vl) else 0.0)
            feat[f'{c}_slope']   = (float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0])
                                    if len(v) > 2 else 0.0)
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan

        feat.update(hrv_time_domain(wa['heart_rate']))
        feat.update(hrv_frequency_domain(wa['heart_rate']))
        feat.update(eda_peak_features(wa['eda']))

        for c in ['heart_rate', 'eda']:
            vs = wshrt[c].dropna().values.astype(float)
            if len(vs) == 0:
                for s in ['short_mean', 'short_std', 'short_max', 'short_slope']:
                    feat[f'{c}_{s}'] = np.nan
            else:
                feat[f'{c}_short_mean']  = float(np.mean(vs))
                feat[f'{c}_short_std']   = float(np.std(vs))
                feat[f'{c}_short_max']   = float(np.max(vs))
                feat[f'{c}_short_slope'] = (float(np.polyfit(np.linspace(0, 1, len(vs)), vs, 1)[0])
                                            if len(vs) > 2 else 0.0)

        for c in ['heart_rate', 'eda', 'temperature']:
            vlag = wlag[c].dropna().values.astype(float)
            cur  = feat.get(f'{c}_mean', np.nan)
            if len(vlag) == 0:
                feat[f'{c}_lag_mean']  = np.nan
                feat[f'{c}_lag_std']   = np.nan
                feat[f'{c}_delta_lag'] = np.nan
            else:
                lag_mean = float(np.mean(vlag))
                feat[f'{c}_lag_mean']  = lag_mean
                feat[f'{c}_lag_std']   = float(np.std(vlag))
                feat[f'{c}_delta_lag'] = (cur - lag_mean) if np.isfinite(cur) else np.nan

        ref = refs.get(pid, {})
        if ref:
            hr_m   = feat.get('heart_rate_mean', np.nan)
            eda_m  = feat.get('eda_mean', np.nan)
            temp_m = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_m   - ref['hr'])                   if np.isfinite(hr_m)   else np.nan
            feat['hr_dev_rest_std']  = (hr_m   - ref['hr'])  / ref['hr_std']  if np.isfinite(hr_m)   else np.nan
            feat['eda_dev_rest']     = (eda_m  - ref['eda'])                  if np.isfinite(eda_m)  else np.nan
            feat['eda_dev_rest_std'] = (eda_m  - ref['eda']) / ref['eda_std'] if np.isfinite(eda_m)  else np.nan
            feat['temp_dev_rest']    = (temp_m - ref['temp'])                 if np.isfinite(temp_m) else np.nan
            hd = feat.get('hr_dev_rest_std', np.nan)
            ed = feat.get('eda_dev_rest_std', np.nan)
            feat['compound_stress']  = (hd + ed) if (np.isfinite(hd) and np.isfinite(ed)) else np.nan
        else:
            for k in ['hr_dev_rest','hr_dev_rest_std','eda_dev_rest',
                      'eda_dev_rest_std','temp_dev_rest','compound_stress']:
                feat[k] = np.nan

        try:
            hr   = wa['heart_rate'].dropna().values.astype(float)
            eda  = wa['eda'].dropna().values.astype(float)
            temp = wa['temperature'].dropna().values.astype(float)
            n_min = min(len(hr), len(eda), len(temp))
            if n_min >= 30:
                h, e, t = hr[:n_min], eda[:n_min], temp[:n_min]
                feat['corr_hr_eda']   = float(np.corrcoef(h, e)[0,1]) if h.std()>1e-6 and e.std()>1e-6 else 0.0
                feat['corr_hr_temp']  = float(np.corrcoef(h, t)[0,1]) if h.std()>1e-6 and t.std()>1e-6 else 0.0
                feat['corr_eda_temp'] = float(np.corrcoef(e, t)[0,1]) if e.std()>1e-6 and t.std()>1e-6 else 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except Exception:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan

        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df_e)} done')

    return pd.DataFrame(rows).set_index('id')


train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}

print('\nExtracting train features...')
train_features = extract_features(TRAIN_LABEL_E, TRAIN_DATA, train_pid_map, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL_E,  TEST_DATA,  train_pid_map, TEST_REFS)
print(f'\ntrain: {train_features.shape}  |  test: {test_features.shape}')


Extracting train features...
  200/815 done
  400/815 done
  600/815 done
  800/815 done
Extracting test features...
  200/1028 done
  400/1028 done
  600/1028 done
  800/1028 done
  1000/1028 done

train: (815, 157)  |  test: (1028, 157)


## 5. PREPARE MATRICES

In [7]:
tli        = TRAIN_LABEL.set_index('id')
y          = tli.loc[train_features.index, 'stress'].astype(int)
pid_groups = tli.loc[train_features.index, 'pid']

common_cols    = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),      columns=common_cols, index=test_features.index)

counts        = Counter(y)
total         = len(y)
# Cap class-1 weight more aggressively — was over-boosting minority
class_weights  = {0: total / (3 * counts[0]),
                  1: min(total / (3 * counts[1]), 2.0),   # tighter cap vs v23's 2.5
                  2: total / (3 * counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior    = np.array([counts[i] / total for i in range(3)])

print(f'\nX_imp        : {X_imp.shape}')
print(f'Class counts : {dict(counts)}')
print(f'Class weights: {dict((k, round(v, 3)) for k, v in class_weights.items())}')
print(f'Train prior  : {train_prior.round(3).tolist()}')


X_imp        : (815, 157)
Class counts : {1: 66, 0: 162, 2: 587}
Class weights: {0: 1.677, 1: 2.0, 2: 0.463}
Train prior  : [0.199, 0.081, 0.72]


## 6. LGBM PARAMS

In [8]:
LGBM_PARAMS = dict(
    n_estimators=1200, learning_rate=0.015, num_leaves=127, max_depth=-1,
    min_child_samples=15, subsample=0.65, colsample_bytree=0.55,
    reg_alpha=0.2, reg_lambda=0.5,
    class_weight='balanced', objective='multiclass', num_class=3,
    n_jobs=-1, verbose=-1,
)

## 7. LOPO — HONEST OOF PROBABILITIES

Used for threshold tuning & constrained alpha search (NO smoothing here)

In [9]:
print('\n══ LOPO (honest OOF) ══')
lopo_oof_proba = np.zeros((len(X_imp), 3))

for tr_idx, val_idx in LeaveOneGroupOut().split(X_imp, y, pid_groups):
    pid_val = pid_groups.iloc[val_idx[0]]
    model   = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': RANDOM_SEED})
    model.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        sample_weight=sample_weights[tr_idx],
        eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(-1)],
    )
    lopo_oof_proba[val_idx] = model.predict_proba(X_imp.iloc[val_idx])
    pred = np.argmax(lopo_oof_proba[val_idx], axis=1)
    ba   = balanced_accuracy_score(y.iloc[val_idx], pred)
    print(f'  PID {pid_val}: BA={ba:.4f}  dist={dict(Counter(pred))}')

y_np        = y.values.astype(int)
lopo_ba_raw = balanced_accuracy_score(y_np, np.argmax(lopo_oof_proba, axis=1))
raw_dist    = dict(Counter(np.argmax(lopo_oof_proba, axis=1)))
print(f'\nHonest LOPO BA (raw argmax): {lopo_ba_raw:.4f}')
print(f'LOPO raw class dist:         {raw_dist}')


# ══════════════════════════════════════════════════════════════════════════════
# 8. THRESHOLD TUNING ON LOPO OOF  ← PRIMARY CALIBRATION METHOD
#
#    Instead of scaling priors, we directly search decision thresholds
#    [t0, t1, t2] where predicted class = argmax(p * [1/t0, 1/t1, 1/t2]).
#    Constrain: predicted class distribution must stay within 1.5× of prior.
#    Optimize: balanced accuracy on LOPO OOF (NO smoothing).
# ══════════════════════════════════════════════════════════════════════════════
print('\n══ Threshold tuning on LOPO OOF ══')

def apply_thresholds(proba, t_vec):
    """Scale columns then argmax — equivalent to moving decision boundaries."""
    scaled = proba * np.array(t_vec)
    return np.argmax(scaled, axis=1)

def dist_ok(preds, prior, total, max_ratio=1.5):
    """Return True if every class fraction stays within max_ratio of prior."""
    cnt   = np.bincount(preds, minlength=3)
    fracs = cnt / len(preds)
    # Allow class to be anywhere between prior/max_ratio and prior*max_ratio
    for i in range(3):
        if prior[i] > 0:
            if fracs[i] > prior[i] * max_ratio:
                return False
            if fracs[i] < prior[i] / max_ratio:
                return False
    return True

# Grid search: t_vec = [t0, t1, t2]
# t > 1  → boost that class  (lower threshold to predict it)
# t < 1  → suppress that class
grid = np.array([0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 1.8, 2.2, 2.8, 3.5])

best_thresh_ba  = -np.inf
best_t_vec      = np.array([1.0, 1.0, 1.0])

for t0, t1, t2 in product(grid, grid, grid):
    t_vec = np.array([t0, t1, t2])
    preds = apply_thresholds(lopo_oof_proba, t_vec)
    if not dist_ok(preds, train_prior, total, max_ratio=1.5):
        continue
    ba = balanced_accuracy_score(y_np, preds)
    if ba > best_thresh_ba:
        best_thresh_ba = ba
        best_t_vec     = t_vec

thresh_dist = np.bincount(apply_thresholds(lopo_oof_proba, best_t_vec), minlength=3)
print(f'Best threshold vec : {best_t_vec}')
print(f'LOPO BA (thresh)   : {best_thresh_ba:.4f}  vs raw {lopo_ba_raw:.4f}')
print(f'Thresh class dist  : {thresh_dist.tolist()}  target ~{[int(p*total) for p in train_prior]}')


══ LOPO (honest OOF) ══
  PID 43JW: BA=0.0000  dist={np.int64(1): 71, np.int64(0): 22}
  PID C8Q6: BA=0.5000  dist={np.int64(2): 152}
  PID DT5C: BA=0.3440  dist={np.int64(0): 56, np.int64(1): 34}
  PID F1ZM: BA=0.4739  dist={np.int64(0): 7, np.int64(2): 130}
  PID HDS9: BA=0.3333  dist={np.int64(1): 12, np.int64(0): 123}
  PID P4DZ: BA=0.2418  dist={np.int64(0): 39, np.int64(1): 105}
  PID TPQI: BA=0.2857  dist={np.int64(1): 17, np.int64(0): 47}

Honest LOPO BA (raw argmax): 0.4683
LOPO raw class dist:         {np.int64(0): 294, np.int64(1): 239, np.int64(2): 282}

══ Threshold tuning on LOPO OOF ══
Best threshold vec : [0.3 0.3 1.2]
LOPO BA (thresh)   : 0.2661  vs raw 0.4683
Thresh class dist  : [130, 81, 604]  target ~[162, 66, 587]


## 9. CONSTRAINED ALPHA GRID SEARCH  ← SECONDARY CALIBRATION

Same logic but uses prior^alpha scaling (keeps v23 approach, tightly constrained)

In [10]:
print('\n══ Constrained alpha grid search ══')

alpha_grid = np.array([0.0, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.5])

def calibrate_proba(proba, alpha_vec, prior):
    cal = proba * (prior ** np.array(alpha_vec))
    return cal / (cal.sum(axis=1, keepdims=True) + 1e-9)

best_alpha_ba  = -np.inf
best_alpha_vec = np.array([1.0, 1.0, 1.0])

for a0, a1, a2 in product(alpha_grid, alpha_grid, alpha_grid):
    av    = np.array([a0, a1, a2])
    cal   = calibrate_proba(lopo_oof_proba, av, train_prior)
    preds = np.argmax(cal, axis=1)
    if not dist_ok(preds, train_prior, total, max_ratio=1.5):
        continue
    ba = balanced_accuracy_score(y_np, preds)
    if ba > best_alpha_ba:
        best_alpha_ba  = ba
        best_alpha_vec = av

alpha_dist = np.bincount(
    np.argmax(calibrate_proba(lopo_oof_proba, best_alpha_vec, train_prior), axis=1), minlength=3)
print(f'Best alpha_vec    : {best_alpha_vec}')
print(f'LOPO BA (alpha)   : {best_alpha_ba:.4f}  vs raw {lopo_ba_raw:.4f}')
print(f'Alpha class dist  : {alpha_dist.tolist()}')


══ Constrained alpha grid search ══
Best alpha_vec    : [0.9 0.6 0.3]
LOPO BA (alpha)   : 0.2701  vs raw 0.4683
Alpha class dist  : [161, 65, 589]


## 10. FINAL 7-SEED LGBM ENSEMBLE

In [11]:
SEEDS    = [42, 7, 123, 17, 99, 256, 314]
N_SPLITS = 5

print('\n══ LightGBM: 7 seeds × 5 folds ══')
lgbm_test_proba = np.zeros((len(X_test_imp), 3))
lgbm_cv_scores  = []

for seed in SEEDS:
    skf         = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_proba  = np.zeros((len(X_test_imp), 3))
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        fold_scores.append(balanced_accuracy_score(y.iloc[val_idx], m.predict(X_imp.iloc[val_idx])))
        seed_proba += m.predict_proba(X_test_imp)
    seed_proba /= N_SPLITS
    lgbm_test_proba += seed_proba
    lgbm_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed}: leaky CV={np.mean(fold_scores):.4f}')

lgbm_test_proba /= len(SEEDS)
print(f'\nLGBM mean leaky CV: {np.mean(lgbm_cv_scores):.4f}')


══ LightGBM: 7 seeds × 5 folds ══
  Seed 42: leaky CV=0.9181
  Seed 7: leaky CV=0.9109
  Seed 123: leaky CV=0.9063
  Seed 17: leaky CV=0.8817
  Seed 99: leaky CV=0.8975
  Seed 256: leaky CV=0.8819
  Seed 314: leaky CV=0.9076

LGBM mean leaky CV: 0.9006


## 11. OPTIONAL XGBOOST ENSEMBLE

In [12]:
if HAS_XGB:
    print('\n══ XGBoost: 7 seeds × 5 folds ══')
    XGB_PARAMS = dict(
        n_estimators=600, learning_rate=0.02, max_depth=6,
        subsample=0.7, colsample_bytree=0.6,
        reg_alpha=0.3, reg_lambda=1.0,
        eval_metric='mlogloss',
        early_stopping_rounds=50,  # constructor param for XGBoost >= 2.0
        n_jobs=-1, verbosity=0,
    )
    xgb_test_proba = np.zeros((len(X_test_imp), 3))
    xgb_cv_scores  = []

    for seed in SEEDS:
        skf         = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        seed_proba  = np.zeros((len(X_test_imp), 3))
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
            m = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(
                X_imp.iloc[tr_idx], y.iloc[tr_idx],
                sample_weight=sample_weights[tr_idx],
                eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
                verbose=False,
            )
            fold_scores.append(balanced_accuracy_score(y.iloc[val_idx], m.predict(X_imp.iloc[val_idx])))
            seed_proba += m.predict_proba(X_test_imp)
        seed_proba /= N_SPLITS
        xgb_test_proba += seed_proba
        xgb_cv_scores.append(np.mean(fold_scores))
        print(f'  Seed {seed}: leaky CV={np.mean(fold_scores):.4f}')

    xgb_test_proba /= len(SEEDS)
    print(f'\nXGB mean leaky CV: {np.mean(xgb_cv_scores):.4f}')
    raw_test_proba = 0.60 * lgbm_test_proba + 0.40 * xgb_test_proba
    print('Final proba: 60% LGBM + 40% XGB blend')
else:
    raw_test_proba = lgbm_test_proba
    print('\nUsing LGBM-only probabilities')

raw_argmax_dist = dict(Counter(np.argmax(raw_test_proba, axis=1)))
print(f'Raw test argmax dist: {raw_argmax_dist}')


══ XGBoost: 7 seeds × 5 folds ══
  Seed 42: leaky CV=0.8715
  Seed 7: leaky CV=0.8600
  Seed 123: leaky CV=0.8557
  Seed 17: leaky CV=0.8516
  Seed 99: leaky CV=0.8709
  Seed 256: leaky CV=0.8459
  Seed 314: leaky CV=0.8693

XGB mean leaky CV: 0.8607
Final proba: 60% LGBM + 40% XGB blend
Raw test argmax dist: {np.int64(2): 422, np.int64(1): 209, np.int64(0): 397}


## 12. GENERATE ALL SUBMISSIONS

In [13]:
def save_csv(preds, fname):
    cnt   = np.bincount(preds.astype(int), minlength=3)
    fracs = cnt / len(preds)
    dev   = np.abs(fracs - train_prior).max()
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds.astype(int)}).to_csv(fname, index=False)
    return cnt, fracs, dev

print('\n┌──────────────────────────────────┬──────────────────────┬───────────────┬───────┐')
print('│ File                             │ method               │ dist          │  dev  │')
print('├──────────────────────────────────┼──────────────────────┼───────────────┼───────┤')

# A — Raw argmax (no calibration) — SAFEST FALLBACK
preds_raw = np.argmax(raw_test_proba, axis=1)
c_raw, f_raw, d_raw = save_csv(preds_raw, 'submission_raw.csv')
print(f'│ submission_raw.csv               │ raw argmax           │ {str(c_raw.tolist()):13s} │ {d_raw:.3f} │')

# B — Threshold tuned
preds_thresh = apply_thresholds(raw_test_proba, best_t_vec)
c_thr, f_thr, d_thr = save_csv(preds_thresh, 'submission_thresh.csv')
print(f'│ submission_thresh.csv            │ t={best_t_vec.round(1)}  │ {str(c_thr.tolist()):13s} │ {d_thr:.3f} │  ← RECOMMENDED')

# C — Constrained alpha
cal_test   = calibrate_proba(raw_test_proba, best_alpha_vec, train_prior)
preds_alpha = np.argmax(cal_test, axis=1)
c_alp, f_alp, d_alp = save_csv(preds_alpha, 'submission_opt.csv')
print(f'│ submission_opt.csv               │ alpha={best_alpha_vec.round(1)} │ {str(c_alp.tolist()):13s} │ {d_alp:.3f} │')

# D — Uniform alpha=1.0 (identity, equivalent to raw — sanity check)
cal_id     = calibrate_proba(raw_test_proba, [1.0, 1.0, 1.0], train_prior)
preds_id   = np.argmax(cal_id, axis=1)
c_id, f_id, d_id = save_csv(preds_id, 'submission_alpha_1.0.csv')
print(f'│ submission_alpha_1.0.csv         │ alpha=[1,1,1]         │ {str(c_id.tolist()):13s} │ {d_id:.3f} │')

print('└──────────────────────────────────┴──────────────────────┴───────────────┴───────┘')

# DEFAULT = threshold submission (highest honest LOPO BA + distribution-constrained)
shutil.copy('submission_thresh.csv', 'submission.csv')
print(f'\n>>> DEFAULT submission.csv = threshold-tuned')
print(f'    t_vec       : {best_t_vec}')
print(f'    LOPO BA     : {best_thresh_ba:.4f}  (raw: {lopo_ba_raw:.4f})')
print(f'    dist (0/1/2): {c_thr.tolist()}')
print(f'    fracs       : {f_thr.round(3).tolist()}')
print(f'    train prior : {train_prior.round(3).tolist()}')
print(f'    target      : ~[0.199, 0.081, 0.72]')


┌──────────────────────────────────┬──────────────────────┬───────────────┬───────┐
│ File                             │ method               │ dist          │  dev  │
├──────────────────────────────────┼──────────────────────┼───────────────┼───────┤
│ submission_raw.csv               │ raw argmax           │ [397, 209, 422] │ 0.310 │
│ submission_thresh.csv            │ t=[0.3 0.3 1.2]  │ [152, 87, 789] │ 0.051 │  ← RECOMMENDED
│ submission_opt.csv               │ alpha=[0.9 0.6 0.3] │ [162, 83, 783] │ 0.041 │
│ submission_alpha_1.0.csv         │ alpha=[1,1,1]         │ [195, 43, 790] │ 0.048 │
└──────────────────────────────────┴──────────────────────┴───────────────┴───────┘

>>> DEFAULT submission.csv = threshold-tuned
    t_vec       : [0.3 0.3 1.2]
    LOPO BA     : 0.2661  (raw: 0.4683)
    dist (0/1/2): [152, 87, 789]
    fracs       : [0.148, 0.085, 0.768]
    train prior : [0.199, 0.081, 0.72]
    target      : ~[0.199, 0.081, 0.72]


## 13. FEATURE IMPORTANCE

In [14]:
last_model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
last_model.fit(X_imp, y, sample_weight=sample_weights, callbacks=[lgb.log_evaluation(-1)])
imp = pd.Series(last_model.feature_importances_, index=X_imp.columns).sort_values(ascending=False)

print('\nTop 30 features:')
for name, val in imp.head(30).items():
    print(f'  {name:35s}: {int(val):5d}')


Top 30 features:
  session_total_labels               :  3294
  temp_dev_rest                      :  2196
  pid_enc                            :  1532
  temperature_skew                   :   703
  session_elapsed_ms                 :   650
  eda_dev_rest_std                   :   627
  temperature_lag_mean               :   546
  eda_t1_mean                        :   524
  heart_rate_min                     :   521
  eda_min                            :   470
  temperature_max                    :   459
  eda_skew                           :   456
  compound_stress                    :   444
  eda_dev_rest                       :   423
  eda_range                          :   398
  accel_y_t3t1                       :   394
  accel_z_p10                        :   387
  accel_z_t1_mean                    :   386
  temperature_mean                   :   373
  session_position                   :   372
  accel_mag_mean                     :   369
  temperature_min                    

## 14. SUMMARY & SUBMISSION GUIDE

In [15]:
print('\n══════════════ v24 SUMMARY ══════════════')
print(f'Honest LOPO BA (raw argmax)  : {lopo_ba_raw:.4f}')
print(f'Honest LOPO BA (thresh-tuned): {best_thresh_ba:.4f}')
print(f'Honest LOPO BA (alpha grid)  : {best_alpha_ba:.4f}')
print(f'Leaky SKF CV BA              : {np.mean(lgbm_cv_scores):.4f}  (inflated — ignore)')
print()
print('SUBMISSION ORDER (try in this order if LB keeps dropping):')
print('  1. submission_thresh.csv   ← threshold-tuned, dist-constrained  [START HERE]')
print('  2. submission_opt.csv      ← alpha grid, dist-constrained')
print('  3. submission_raw.csv      ← pure raw argmax, no calibration')
print()

# Decision guide based on LOPO results
if best_thresh_ba >= lopo_ba_raw:
    print('✓ Threshold tuning IMPROVED LOPO BA → submission_thresh.csv is the best bet')
else:
    print('⚠ Threshold tuning did NOT improve LOPO BA → submission_raw.csv may be safer')
    shutil.copy('submission_raw.csv', 'submission.csv')
    print('  DEFAULT switched to submission_raw.csv')
print('═════════════════════════════════════════')


══════════════ v24 SUMMARY ══════════════
Honest LOPO BA (raw argmax)  : 0.4683
Honest LOPO BA (thresh-tuned): 0.2661
Honest LOPO BA (alpha grid)  : 0.2701
Leaky SKF CV BA              : 0.9006  (inflated — ignore)

SUBMISSION ORDER (try in this order if LB keeps dropping):
  1. submission_thresh.csv   ← threshold-tuned, dist-constrained  [START HERE]
  2. submission_opt.csv      ← alpha grid, dist-constrained
  3. submission_raw.csv      ← pure raw argmax, no calibration

⚠ Threshold tuning did NOT improve LOPO BA → submission_raw.csv may be safer
  DEFAULT switched to submission_raw.csv
═════════════════════════════════════════
